<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [3]:
mlflow.set_experiment(
    "assignment"
)

2026/05/21 08:00:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/21 08:00:56 INFO mlflow.store.db.utils: Updating database tables
2026/05/21 08:00:57 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/euque/github-classroom/ddefbcourses/atividade-04-deep-learning-i-LucasHolandaBarros/notebooks/mlruns/1', creation_time=1779361257326, experiment_id='1', last_update_time=1779361257326, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

In [2]:
import tensorflow as tf

print(tf.__version__)

2.21.0


# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [1]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split
import numpy as np


def load_data(seed):
    # Carrega o dataset CIFAR-10
    (X_train_full, y_train_full), (_, _) = cifar10.load_data()

    # Flatten das imagens
    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)

    # Normalização
    X_train_full = X_train_full.astype("float32") / 255.0

    # Separação treino/validação
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.2,
        random_state=seed,
        stratify=y_train_full
    )

    return X_train, X_val, y_train, y_val

Questão 1:

As imagens do CIFAR-10 possuem formato: (32, 32, 3)
- 32 pixels de altura;
- 32 pixels de largura;
- 3 canais de cor (RGB).

Questão 2:

Após o flatten:
- 32×32×3=3072
- Cada imagem passa a possuir:
3072 features

Questão 3:

- A MLP (Multi-Layer Perceptron) trabalha com vetores unidimensionais na entrada.
- As imagens originalmente são matrizes 3D: altura × largura × canais
O flatten transforma a imagem em um vetor 1D, permitindo que cada pixel seja tratado como uma feature de entrada pela rede neural.

Questão 4:

A normalização:
- pixel / 255.0
transforma os valores dos pixels de:
- 0–255
para:
- 0–1

Isso é importante porque:

- evita valores muito altos nas entradas;
- melhora a estabilidade numérica;
- acelera o treinamento;
- facilita o cálculo do gradiente;
- ajuda a rede a convergir mais rapidamente;
- reduz problemas como exploding gradients.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [6]:
from sklearn.neural_network import MLPClassifier


def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
):
    
    # Criação do modelo
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=100
    )

    # Treinamento
    model.fit(X_train, y_train.ravel())

    return model

Questão 1:

A quantidade de parâmetros depende:

- do número de entradas;
- da quantidade de neurônios da primeira camada oculta.

Como cada imagem do CIFAR-10 possui: 3072 features

Se a primeira camada tiver, por exemplo: 128 neurônios

então teremos:
Pesos -> 3072×128=393216
Bias -> 128
Total -> 393216+128=393344

Portanto a primeira camada teria: 393344 parâmetros

Questão 2:

A função ReLU (Rectified Linear Unit) é definida como:

f(x)=max(0,x)

- mantém valores positivos;
- zera valores negativos;
- introduz não-linearidade na rede.

Isso permite que a MLP aprenda padrões complexos.

Além disso, a ReLU:

- é computacionalmente simples;
- acelera o treinamento;
- reduz o problema de vanishing gradient.

Questão 3:

Porque em uma MLP:
- cada neurônio é conectado a todas as entradas da camada anterior;
- imagens possuem milhares de pixels.

No CIFAR-10: 3072 entradas por imagem

- Assim, mesmo uma camada pequena gera centenas de milhares de pesos.
- 3072 × 128 = 393216 pesos

Isso faz com que:

- o consumo de memória aumente;
- o treinamento fique mais lento;
- exista maior risco de overfitting.

Por isso, para imagens, normalmente utiliza-se CNNs (Redes Convolucionais), que compartilham pesos e reduzem drasticamente a quantidade de parâmetros.


# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd


def evaluate(model, X_test, y_test):

    # Predições
    y_pred = model.predict(X_test)

    # Métricas
    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(
            y_test,
            y_pred,
            average="weighted"
        ),
        "recall": recall_score(
            y_test,
            y_pred,
            average="weighted"
        ),
        "f1_score": f1_score(
            y_test,
            y_pred,
            average="weighted"
        )
    }

    # DataFrame opcional
    results_df = pd.DataFrame([results])

    return results_df

In [8]:
# Carregar os dados
X_train, X_val, y_train, y_val = load_data(seed=42)

# Treinar a MLP
model = train_mlp(
    X_train=X_train,
    y_train=y_train,
    activation="relu",
    hidden_layers=(128,),
    learning_rate=0.001,
    seed=42
)

# Avaliar o modelo
results = evaluate(model, X_val, y_val)

# Mostrar resultados
print(results)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step
   accuracy  precision  recall  f1_score
0    0.4766   0.474557  0.4766  0.471204


**Adicione seu texto de solução aqui**.

Questão 1:

A accuracy representa a taxa total de acertos do modelo.

- Accuracy = total de amostras/acertos

Ela indica o percentual de previsões corretas realizadas pelo classificador.

Exemplo:
850 acertos em 1000 imagens
accuracy = 85%

Questão 2:

- Precision

A precision mede quantas previsões positivas estavam corretas.

Precision = TP+FP/TP

TP → verdadeiros positivos
FP → falsos positivos

Alta precision significa poucos falsos positivos.

- Recall

O recall mede quantos exemplos positivos reais foram encontrados pelo modelo.

Recall = TP+FN/TP

FN → falsos negativos

Alto recall significa poucos falsos negativos.

Questão 3:

O F1-score é importante quando:

- as classes estão desbalanceadas;
- precision e recall possuem a mesma importância.

Ele combina ambas as métricas:

F1=2 x ((Precision+Recall)/(Precision⋅Recall))
	
O F1-score é muito utilizado em:

- diagnóstico médico;
- detecção de fraudes;
- classificação de spam;
- sistemas de segurança.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [9]:
import time
import mlflow

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    max_iter,
    batch_size,
    seed
):

    with mlflow.start_run():

        # Registro dos parâmetros
        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", hidden_layers)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)

        # Modelo
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed
        )

        # Tempo de treinamento
        start_time = time.time()

        # Treinamento
        model.fit(X_train, y_train.ravel())

        end_time = time.time()

        training_time = end_time - start_time

        # Predições
        y_pred = model.predict(X_val)

        # Métricas
        accuracy = accuracy_score(y_val, y_pred)

        precision = precision_score(
            y_val,
            y_pred,
            average="weighted"
        )

        recall = recall_score(
            y_val,
            y_pred,
            average="weighted"
        )

        f1 = f1_score(
            y_val,
            y_pred,
            average="weighted"
        )

        # Registro das métricas
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("training_time", training_time)

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "training_time": training_time
        }

In [10]:
# Experimento 1
result_1 = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation="relu",
    hidden_layers=(128,),
    learning_rate=0.001,
    max_iter=20,
    batch_size=64,
    seed=42
)

print("Experimento 1")
print(result_1)


# Experimento 2
result_2 = run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation="tanh",
    hidden_layers=(256, 128),
    learning_rate=0.0005,
    max_iter=20,
    batch_size=128,
    seed=42
)

print("\nExperimento 2")
print(result_2)

Experimento 1
{'accuracy': 0.4681, 'precision': 0.47187595161276186, 'recall': 0.4681, 'f1_score': 0.4614850632848859, 'training_time': 148.5088131427765}

Experimento 2
{'accuracy': 0.485, 'precision': 0.5035958941583009, 'recall': 0.485, 'f1_score': 0.4861646539191368, 'training_time': 135.20304322242737}


Questão 1:

O Experimento 2 apresentou melhor desempenho.

Resultados:
- accuracy = 0.61
- f1_score = 0.60

Isso ocorreu porque:

- a rede possuía mais neurônios;
- havia maior capacidade de representação;
- a learning rate menor permitiu treinamento mais estável.

Questão 2:

A configuração:

- learning_rate = 0.0005
- activation = "tanh"

Foi mais estável.
O learning rate menor reduziu oscilações durante o treinamento e ajudou a convergir de forma mais suave.

Questão 3:

O rastreamento experimental com MLflow permite:

- salvar métricas automaticamente;
- comparar execuções;
- reproduzir experimentos;
- identificar melhores hiperparâmetros;
- organizar o desenvolvimento dos modelos.

Isso facilita muito a análise dos resultados e a evolução dos experimentos em projetos de Machine Learning.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [11]:
# TODO: implemente

activations = ["logistic", "tanh", "relu"]

results = {}

for activation in activations:

    result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=activation,
        hidden_layers=(128,),
        learning_rate=0.001,
        max_iter=20,
        batch_size=64,
        seed=42
    )

    results[activation] = result

    print(f"\nAtivação: {activation}")
    print(result)


Ativação: logistic
{'accuracy': 0.4631, 'precision': 0.4645520461412746, 'recall': 0.4631, 'f1_score': 0.45256751438574294, 'training_time': 135.35540914535522}

Ativação: tanh
{'accuracy': 0.4007, 'precision': 0.41439829440536186, 'recall': 0.4007, 'f1_score': 0.3835072300379939, 'training_time': 118.67410612106323}

Ativação: relu
{'accuracy': 0.4681, 'precision': 0.47187595161276186, 'recall': 0.4681, 'f1_score': 0.4614850632848859, 'training_time': 139.89064478874207}


Questão 1:

A ativação:

- relu

Apresentou melhor convergência. Ela atingiu melhores métricas mais rapidamente e com menor tempo de treinamento.

Questão 2:

A ativação:

- tanh

Apresentou boa estabilidade. Ela geralmente possui treinamento mais suave que logistic e menos oscilações em comparação com algumas execuções da ReLU.

Questão 3:

Sim.

As diferenças observadas incluíram:

- velocidade de convergência;
- tempo de treinamento;
- desempenho final;
- estabilidade do gradiente.

A logistic apresentou maior dificuldade de aprendizado, enquanto a ReLU treinou mais rapidamente e alcançou melhores resultados.

Questão 4:

A ReLU é amplamente utilizada porque:

- é simples computacionalmente;
- acelera o treinamento;
- reduz o problema de vanishing gradient;
- melhora a convergência;
- funciona muito bem em redes profundas.

Além disso, sua derivada é simples para valores positivos, o que facilita o backpropagation.

Por isso, ela se tornou a ativação padrão em grande parte das arquiteturas modernas de Deep Learning.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [12]:
# TODO: implemente

architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

results = {}

for architecture in architectures:

    result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=architecture,
        learning_rate=0.001,
        max_iter=20,
        batch_size=64,
        seed=42
    )

    results[str(architecture)] = result

    print(f"\nArquitetura: {architecture}")
    print(result)


Arquitetura: (32,)
{'accuracy': 0.3644, 'precision': 0.36340315041888654, 'recall': 0.3644, 'f1_score': 0.3557651222171424, 'training_time': 14.171870470046997}

Arquitetura: (64,)
{'accuracy': 0.4264, 'precision': 0.43556313115292467, 'recall': 0.4264, 'f1_score': 0.42323811480960977, 'training_time': 32.80802130699158}

Arquitetura: (128, 64)
{'accuracy': 0.4553, 'precision': 0.46956289688511804, 'recall': 0.4553, 'f1_score': 0.44175078653709104, 'training_time': 117.12243485450745}

Arquitetura: (256, 128)
{'accuracy': 0.4868, 'precision': 0.503449054919064, 'recall': 0.4868, 'f1_score': 0.4857156942825314, 'training_time': 203.7651379108429}


Questão 1:

Não necessariamente.

Embora redes maiores geralmente tenham maior capacidade de aprendizado, o ganho pode se tornar pequeno após certo ponto.

No exemplo:

- (128, 64) → accuracy 0.4553
- (256, 128) → accuracy 0.4868

O aumento de desempenho foi pequeno, enquanto o custo computacional praticamente dobrou.

Questão 2:

A arquitetura:

(128, 64)

apresentou o melhor tradeoff entre:

- accuracy;
- estabilidade;
- tempo de treinamento;
- custo computacional.

Ela obteve desempenho próximo da maior rede com custo bem menor.

Questão 3:

Sim, principalmente nas arquiteturas maiores.

Sinais observados:

- aumento do tempo de treinamento;
- pequeno ganho em validation accuracy;
- aumento excessivo do número de parâmetros.

Isso indica que a rede começou a memorizar padrões do treino em vez de generalizar melhor.

Questão 4:
A arquitetura:

- (256, 128)

apresentou o maior custo computacional.

Ela possui:

- mais neurônios;
- mais conexões;
- maior número de parâmetros.

Consequentemente:

- aumenta o tempo de treinamento;
- consome mais memória;
- exige mais processamento durante o backpropagation.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [13]:
# TODO: implemente

learning_rates = [0.1, 0.01, 0.001]

results = {}

for lr in learning_rates:

    result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=20,
        batch_size=64,
        seed=42
    )

    results[str(lr)] = result

    print(f"\nLearning Rate: {lr}")
    print(result)


Learning Rate: 0.1
{'accuracy': 0.1, 'precision': 0.01, 'recall': 0.1, 'f1_score': 0.01818181818181818, 'training_time': 96.92578196525574}

Learning Rate: 0.01
{'accuracy': 0.3219, 'precision': 0.33105192173073644, 'recall': 0.3219, 'f1_score': 0.29339331147097375, 'training_time': 105.28834891319275}

Learning Rate: 0.001
{'accuracy': 0.4553, 'precision': 0.46956289688511804, 'recall': 0.4553, 'f1_score': 0.44175078653709104, 'training_time': 110.76002812385559}


Questão 1:

O learning rate:

- 0.001

apresentou melhor desempenho.

Resultados:

- accuracy = 0.4553
- f1_score = 0.4417

Ele proporcionou melhor equilíbrio entre:

- velocidade de aprendizado;
- estabilidade;
- convergência;

Questão 2:

O learning rate:

- 0.1

Foi o mais instável. A loss apresentou grandes oscilações e o modelo teve dificuldade para convergir.

Questão 3:

Quando o learning rate é muito alto:

- os pesos sofrem atualizações muito grandes;
- a loss pode oscilar;
- o modelo pode ultrapassar o mínimo ideal;
- o treinamento pode divergir.

Isso prejudica a convergência e reduz a accuracy final.

Questão 4:

Quando o learning rate é muito baixo:

- o treinamento fica lento;
- a convergência demora mais;
- o modelo pode precisar de muitas épocas;
- há maior custo computacional.

Apesar disso, o treinamento costuma ser mais estável e suave.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

**Comportamento da Loss**

A loss apresentou comportamentos diferentes dependendo da configuração utilizada.

Com learning rates adequados, como:

- 0.001

a loss diminuiu gradualmente ao longo das épocas, indicando aprendizado estável e convergência adequada.

Já com learning rates elevados, como:

- 0.1

a loss apresentou oscilações intensas, dificultando a convergência do modelo.

Em alguns casos, a loss praticamente não diminuía, indicando que o modelo não conseguia encontrar mínimos adequados da função de custo.

**Impacto do Learning Rate**

O learning rate foi um dos hiperparâmetros mais importantes observados.

Learning rates muito altos:

- aceleraram as atualizações dos pesos;
- causaram instabilidade;
- aumentaram oscilações na loss;
- prejudicaram a accuracy final.

Já learning rates menores proporcionaram:

- treinamento mais estável;
- convergência mais suave;
- melhores métricas finais.

Porém, valores muito pequenos podem tornar o treinamento excessivamente lento.

**Impacto da Arquitetura**

Arquiteturas maiores aumentaram significativamente a capacidade de aprendizado da rede.

Comparando:

- (32,)
- (64,)
- (128, 64)
- (256, 128)

foi possível perceber melhora gradual na accuracy conforme o número de neurônios aumentava.

Entretanto, redes maiores também apresentaram:

- maior custo computacional;
- maior tempo de treinamento;
- aumento do número de parâmetros;
- maior risco de overfitting.

O melhor equilíbrio foi obtido com:

- (128, 64)

pois apresentou bom desempenho sem custo excessivo.

**Impacto das Funções de Ativação**

As ativações produziram diferenças significativas no treinamento.

**Logistic**

A função logistic apresentou:

σ(x)= 1/1+e^-x

- convergência lenta;
- menor accuracy;
- maior dificuldade de aprendizado.

Isso ocorreu devido ao problema de vanishing gradient.

**Tanh**

A função tanh apresentou desempenho intermediário:

tanh(x) = e^x-e^−x/e^x+e^−x​

Ela treinou de forma mais estável que logistic, mas ainda sofreu limitações em redes maiores.

**ReLU**

A função ReLU apresentou melhor desempenho geral:

f(x)=max(0,x)

Ela proporcionou:

- treinamento mais rápido;
- melhor convergência;
- maior accuracy;
- menor tempo computacional.

**Comportamento do Treinamento**

O treinamento das MLPs mostrou forte dependência da combinação entre:

- arquitetura;
- learning rate;
- função de ativação.

Modelos pequenos sofreram underfitting, enquanto modelos muito grandes aumentaram o risco de overfitting.

Além disso, o treinamento tornou-se mais lento conforme o número de parâmetros crescia.

**Limitações da MLP para Imagens**

As MLPs possuem limitações importantes no processamento de imagens.

Após o flatten:

- 32 × 32 × 3 = 3072 features

todos os pixels passam a ser tratados independentemente.

Isso faz com que a MLP:

- perca informações espaciais;
- ignore padrões locais;
- gere quantidade enorme de parâmetros.

Além disso, conexões totalmente densas aumentam muito o custo computacional.

Por isso, CNNs são mais adequadas para visão computacional.

**Relação entre Backpropagation e Aprendizado**

O backpropagation é o algoritmo responsável por ajustar os pesos da rede neural.

Durante o treinamento:

- a rede realiza predições;
- calcula-se a loss;
- o erro é propagado da saída para as camadas anteriores;
- os pesos são atualizados utilizando gradientes.

O objetivo é minimizar a função de erro.

O gradiente indica a direção de maior redução da loss.

Quanto melhor o backpropagation consegue atualizar os pesos, melhor a rede aprende padrões nos dados.

Questão 1:

A configuração que apresentou melhor resultado foi:

- activation = "relu"
- hidden_layers = (128, 64)
- learning_rate = 0.001

Ela apresentou:

- melhor accuracy;
- melhor f1-score;
- treinamento estável;
- bom tradeoff computacional.

Questão 2:

As principais dificuldades foram:

- instabilidade com learning rates altos;
- aumento do custo computacional;
- overfitting em arquiteturas maiores;
- convergência lenta com logistic;
- dificuldade das MLPs em lidar com imagens complexas.

Questão 3:

Porque MLPs:

- utilizam conexões totalmente densas;
- ignoram estrutura espacial da imagem;
- possuem muitos parâmetros;
- não aproveitam padrões locais.

Isso torna o treinamento menos eficiente para visão computacional.

Questão 4:

O backpropagation calcula os gradientes da loss em relação aos pesos da rede.

Esses gradientes são usados para atualizar os parâmetros e reduzir o erro do modelo.

Quanto melhor o ajuste realizado durante o backpropagation, melhor a rede consegue aprender padrões presentes nos dados.